In [25]:
import os
import io
import json
import glob
import math
from dataclasses import dataclass
from typing import Dict, Any, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import zstandard as zstd


In [26]:

# =========================
# I/O: stream read jsonl.zst
# =========================
def iter_jsonl_zst(path: str, verbose: bool = True) -> Iterable[Dict[str, Any]]:
    """Yield dict rows from a .jsonl.zst file without loading everything into memory.
    Skip malformed JSON lines instead of crashing the whole job.
    """
    bad_lines = 0

    with open(path, "rb") as f:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(f) as reader:
            text = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")

            for line_no, line in enumerate(text, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    yield json.loads(line)
                except json.JSONDecodeError as e:
                    bad_lines += 1
                    if verbose:
                        preview = line[:300].replace("\n", " ")
                        print(
                            f"[BAD JSON] file={path} line={line_no} "
                            f"err={e} preview={preview}"
                        )
                    continue
                except Exception as e:
                    bad_lines += 1
                    if verbose:
                        preview = line[:300].replace("\n", " ")
                        print(
                            f"[BAD LINE] file={path} line={line_no} "
                            f"err={type(e).__name__}: {e} preview={preview}"
                        )
                    continue

    if bad_lines > 0 and verbose:
        print(f"[WARN] file={path} skipped_bad_lines={bad_lines}")
                    
                    


# =========================
# Helpers
# =========================
def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

def parse_freq_ms(freq_min: int) -> int:
    return int(freq_min) * 60_000

def floor_bucket_ms(ts_ms: int, freq_ms: int) -> int:
    return (ts_ms // freq_ms) * freq_ms

def ms_to_utc_datetime(ms: int) -> pd.Timestamp:
    return pd.to_datetime(ms, unit="ms", utc=True)


# =========================
# Stage 1: Tick -> Buckets (stream aggregation)
# =========================
@dataclass
class AggTradeBucket:
    last_price: float = np.nan
    sum_qty: float = 0.0
    sum_notional: float = 0.0
    n_trades: int = 0
    buy_qty: float = 0.0   # m==False
    sell_qty: float = 0.0  # m==True

    def update(self, price: float, qty: float, m: bool):
        self.last_price = price
        self.sum_qty += qty
        self.sum_notional += price * qty
        self.n_trades += 1
        if m is True:
            self.sell_qty += qty
        else:
            self.buy_qty += qty

    def to_row(self) -> Dict[str, Any]:
        vwap = (self.sum_notional / self.sum_qty) if self.sum_qty > 0 else np.nan
        buy_ratio = (self.buy_qty / self.sum_qty) if self.sum_qty > 0 else np.nan
        return {
            "px_last": self.last_price,
            "px_vwap": vwap,
            "vol_qty": self.sum_qty,
            "vol_notional": self.sum_notional,
            "n_trades": self.n_trades,
            "buy_qty": self.buy_qty,
            "sell_qty": self.sell_qty,
            "buy_ratio": buy_ratio,
        }


@dataclass
class BookTickerBucket:
    # mean stats + last snapshot within bucket
    sum_spread: float = 0.0
    sum_rel_spread: float = 0.0
    sum_imb: float = 0.0
    n: int = 0

    last_mid: float = np.nan
    last_imb: float = np.nan

    def update(self, bid: float, ask: float, bidq: float, askq: float):
        mid = (bid + ask) / 2.0
        spread = ask - bid
        rel_spread = spread / mid if mid != 0 else np.nan
        denom = (bidq + askq)
        imb = (bidq - askq) / denom if denom != 0 else np.nan

        if not np.isnan(spread):
            self.sum_spread += spread
        if not np.isnan(rel_spread):
            self.sum_rel_spread += rel_spread
        if not np.isnan(imb):
            self.sum_imb += imb
        self.n += 1

        self.last_mid = mid
        self.last_imb = imb

    def to_row(self) -> Dict[str, Any]:
        if self.n <= 0:
            return {
                "mid_last": np.nan,
                "spread_mean": np.nan,
                "rel_spread_mean": np.nan,
                "imbalance_mean": np.nan,
                "imbalance_last": np.nan,
            }
        return {
            "mid_last": self.last_mid,
            "spread_mean": self.sum_spread / self.n,
            "rel_spread_mean": self.sum_rel_spread / self.n,
            "imbalance_mean": self.sum_imb / self.n,
            "imbalance_last": self.last_imb,
        }


def aggregate_aggtrade_file(path: str, freq_ms: int) -> Dict[int, AggTradeBucket]:
    buckets: Dict[int, AggTradeBucket] = {}
    for obj in iter_jsonl_zst(path):
        # expected: T,p,q,m
        T = obj.get("T")
        p = obj.get("p")
        q = obj.get("q")
        m = obj.get("m")
        if T is None or p is None or q is None or m is None:
            continue
        try:
            ts_ms = int(T)
            price = float(p)
            qty = float(q)
            mm = bool(m)
        except Exception:
            continue

        b = floor_bucket_ms(ts_ms, freq_ms)
        if b not in buckets:
            buckets[b] = AggTradeBucket()
        buckets[b].update(price, qty, mm)
    return buckets


def aggregate_bookticker_file(path: str, freq_ms: int) -> Dict[int, BookTickerBucket]:
    buckets: Dict[int, BookTickerBucket] = {}
    seen = 0
    used = 0

    for obj in iter_jsonl_zst(path):
        seen += 1

        # futures/spot 兼容：优先 event_time，其次 T / E
        t = obj.get("event_time", obj.get("T", obj.get("E")))
        if t is None:
            continue

        # accept both naming conventions
        bid_s = obj.get("best_bid_price", obj.get("b"))
        ask_s = obj.get("best_ask_price", obj.get("a"))
        bidq_s = obj.get("best_bid_qty", obj.get("B"))
        askq_s = obj.get("best_ask_qty", obj.get("A"))
        if bid_s is None or ask_s is None or bidq_s is None or askq_s is None:
            continue

        try:
            ts_ms = int(t)
            bid = float(bid_s)
            ask = float(ask_s)
            bidq = float(bidq_s)
            askq = float(askq_s)
        except Exception:
            continue

        b = floor_bucket_ms(ts_ms, freq_ms)
        if b not in buckets:
            buckets[b] = BookTickerBucket()
        buckets[b].update(bid, ask, bidq, askq)
        used += 1

    print(f"[BOOK] file={path} seen={seen} used={used} buckets={len(buckets)}")
    return buckets


def build_buckets_for_day(day_dir: str, freq_min: int) -> pd.DataFrame:
    """
    day_dir: .../YYYYMMDD
    structure expected: day_dir/00/aggtrade/*.jsonl.zst, day_dir/00/bookticker/*.jsonl.zst, etc
    returns buckets dataframe with timestamp column. if no data found, returns EMPTY df with columns.
    """
    freq_ms = parse_freq_ms(freq_min)

    agg_all: Dict[int, AggTradeBucket] = {}
    book_all: Dict[int, BookTickerBucket] = {}

    hour_dirs = sorted([p for p in glob.glob(os.path.join(day_dir, "*")) if os.path.isdir(p)])

    # diagnostics
    n_agg_files = 0
    n_book_files = 0

    for hdir in hour_dirs:
        agg_files = sorted(glob.glob(os.path.join(hdir, "aggtrade", "*.jsonl.zst")))
        book_files = sorted(glob.glob(os.path.join(hdir, "bookticker", "*.jsonl.zst")))
        n_agg_files += len(agg_files)
        n_book_files += len(book_files)

        # aggtrade
        for f in agg_files:
            agg_b = aggregate_aggtrade_file(f, freq_ms)
            for k, v in agg_b.items():
                if k not in agg_all:
                    agg_all[k] = v
                else:
                    a = agg_all[k]
                    a.last_price = v.last_price if not np.isnan(v.last_price) else a.last_price
                    a.sum_qty += v.sum_qty
                    a.sum_notional += v.sum_notional
                    a.n_trades += v.n_trades
                    a.buy_qty += v.buy_qty
                    a.sell_qty += v.sell_qty

        # bookticker
        for f in book_files:
            book_b = aggregate_bookticker_file(f, freq_ms)
            for k, v in book_b.items():
                if k not in book_all:
                    book_all[k] = v
                else:
                    b = book_all[k]
                    b.sum_spread += v.sum_spread
                    b.sum_rel_spread += v.sum_rel_spread
                    b.sum_imb += v.sum_imb
                    b.n += v.n
                    b.last_mid = v.last_mid if not np.isnan(v.last_mid) else b.last_mid
                    b.last_imb = v.last_imb if not np.isnan(v.last_imb) else b.last_imb

    keys = sorted(set(agg_all.keys()) | set(book_all.keys()))

    # if empty, return empty df with columns
    cols = ["bucket_ms", "timestamp"]
    cols += list(AggTradeBucket().to_row().keys())
    cols += list(BookTickerBucket().to_row().keys())

    if len(keys) == 0:
        print(
            f"[Stage1] SKIP empty day={os.path.basename(day_dir)} "
            f"(hour_dirs={len(hour_dirs)} agg_files={n_agg_files} book_files={n_book_files})"
        )
        return pd.DataFrame(columns=cols)

    rows = []
    for k in keys:
        row = {"bucket_ms": k, "timestamp": ms_to_utc_datetime(k)}

        if k in agg_all:
            row.update(agg_all[k].to_row())
        else:
            row.update({c: np.nan for c in AggTradeBucket().to_row().keys()})

        if k in book_all:
            row.update(book_all[k].to_row())
        else:
            row.update({c: np.nan for c in BookTickerBucket().to_row().keys()})

        rows.append(row)

    df = pd.DataFrame(rows, columns=cols).sort_values("timestamp").reset_index(drop=True)
    return df


def stage1_build_buckets(
    base_dir: str,
    out_dir: str,
    freq_min: int,
    dates: List[str] | None = None,
) -> None:
    """
    base_dir: /Volumes/profit/bitcoin_ticks/spot/btcusdt
    out_dir:  /Volumes/profit/feature_store/buckets
    dates:    optional list like ["20251101","20251102"] to process only those days
    writes:   out_dir/freq=5min/date=YYYYMMDD/buckets.parquet
    """
    ensure_dir(out_dir)

    if dates is None:
        day_dirs = sorted([p for p in glob.glob(os.path.join(base_dir, "*")) if os.path.isdir(p)])
    else:
        day_dirs = [os.path.join(base_dir, d) for d in dates]

    for day_dir in day_dirs:
        date = os.path.basename(day_dir)
        if not os.path.isdir(day_dir):
            print(f"[Stage1] SKIP missing day_dir={day_dir}")
            continue
        if not date.isdigit():
            continue

        df = build_buckets_for_day(day_dir, freq_min=freq_min)
        if df.empty:
            continue

        out_path = os.path.join(out_dir, f"freq={freq_min}min", f"date={date}", "buckets.parquet")
        ensure_dir(os.path.dirname(out_path))
        df.to_parquet(out_path, index=False)
        print(f"[Stage1] wrote {out_path} rows={len(df)}")


# =========================
# Stage 2: Buckets -> Features + label (RV horizon)
# =========================
def build_features_from_buckets(
    df: pd.DataFrame,
    freq_min: int,
    horizon_min: int,
    windows_min: List[int],
) -> pd.DataFrame:
   # df = df.sort_values("timestamp").reset_index(drop=True)
    df = (
        df.sort_values("timestamp")
          .drop_duplicates(subset=["timestamp"], keep="last")
          .reset_index(drop=True)
    )
    
    for c in ["mid_last", "spread_mean", "rel_spread_mean", "imbalance_mean", "imbalance_last"]:
        if c in df.columns:
            df[c] = df[c].ffill(limit=2)

    # price for returns: prefer mid, fallback to px_last
    px = df["mid_last"].copy()
    px = px.fillna(df["px_last"]).ffill()

    df["ret"] = np.log(px).diff()
    df["ret2"] = df["ret"] ** 2

    step = int(freq_min)
    H = int(round(horizon_min / step))
    if H <= 0:
        raise ValueError("horizon_min must be >= freq_min")

    # forward RV horizon: sum ret2 over next H steps
    df["rv_fwd"] = df["ret2"].rolling(H).sum().shift(-H)
    df["y"] = np.log(df["rv_fwd"] + 1e-12)

    # HAR-like past RV
    for wmin in windows_min:
        W = int(round(wmin / step))
        if W > 0:
            df[f"rv_{wmin}m_past"] = df["ret2"].rolling(W).sum()

    # extra rolling microstructure/flow
    for wmin in [5, 15, 60, 240]:
        if wmin < step:
            continue
        W = int(round(wmin / step))
        if "rel_spread_mean" in df.columns:
            df[f"spread_{wmin}m"] = df["rel_spread_mean"].rolling(W).mean()
        if "imbalance_mean" in df.columns:
            df[f"imb_{wmin}m"] = df["imbalance_mean"].rolling(W).mean()
        if "vol_notional" in df.columns:
            df[f"vol_{wmin}m"] = df["vol_notional"].rolling(W).sum()
        if "n_trades" in df.columns:
            df[f"ntr_{wmin}m"] = df["n_trades"].rolling(W).sum()

    # seasonality UTC
    minute_of_day = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["sin_tod"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
    df["cos_tod"] = np.cos(2 * np.pi * minute_of_day / 1440.0)

    return df


def stage2_build_features(
    buckets_dir: str,
    out_dir: str,
    freq_min: int,
    horizon_min: int,
    dates: List[str] | None = None,
) -> None:
    """
    reads:  buckets_dir/freq=5min/date=YYYYMMDD/buckets.parquet
    writes: out_dir/freq=5min/h=60min/date=YYYYMMDD/features.parquet
    dates: optional list for subset
    """
    ensure_dir(out_dir)

    if dates is None:
        pat = os.path.join(buckets_dir, f"freq={freq_min}min", "date=*", "buckets.parquet")
        files = sorted(glob.glob(pat))
    else:
        files = []
        for d in dates:
            f = os.path.join(buckets_dir, f"freq={freq_min}min", f"date={d}", "buckets.parquet")
            if os.path.exists(f):
                files.append(f)
        files = sorted(files)

    if not files:
        raise FileNotFoundError("No bucket parquet files found for given settings.")

    # windows you can tune freely
    windows_min = [5, 15, 60, 240]

    # stitch all (bucket is small) to ensure rolling continuity across days
    all_df = [pd.read_parquet(f) for f in files]
    full = pd.concat(all_df, ignore_index=True).sort_values("timestamp").reset_index(drop=True)

    feat = build_features_from_buckets(full, freq_min=freq_min, horizon_min=horizon_min, windows_min=windows_min)

    # write per day
    feat["date"] = feat["timestamp"].dt.strftime("%Y%m%d")
    for date, g in feat.groupby("date", sort=True):
        # 只写你想要的日期（如果指定了 dates）
        if dates is not None and date not in dates:
            continue
        out_path = os.path.join(out_dir, f"freq={freq_min}min", f"h={horizon_min}min", f"date={date}", "features.parquet")
        ensure_dir(os.path.dirname(out_path))
        g = g.drop(columns=["date"])
        g.to_parquet(out_path, index=False)
        print(f"[Stage2] wrote {out_path} rows={len(g)}")
        
        
def list_available_days(base_dir: str):
    return sorted([os.path.basename(p) for p in glob.glob(os.path.join(base_dir, "*")) if os.path.isdir(p)])

def list_built_days(buckets_out: str, freq_min: int):
    pat = os.path.join(buckets_out, f"freq={freq_min}min", "date=*", "buckets.parquet")
    return sorted({p.split("date=")[-1].split(os.sep)[0] for p in glob.glob(pat)})

def compute_missing_days(base_dir: str, buckets_out: str, freq_min: int):
    have_raw = set(d for d in list_available_days(base_dir) if d.isdigit())
    have_buckets = set(list_built_days(buckets_out, freq_min))
    return sorted(have_raw - have_buckets)


# =========================
# Example run
# =========================
if __name__ == "__main__":
    BASE_DIR = "/Volumes/profit/bitcoin_ticks/futures/btcusdt" # "/Volumes/profit/bitcoin_ticks/spot/btcusdt"
    BUCKETS_OUT = "/Volumes/profit/feature_store_futures/buckets" # "/Volumes/profit/feature_store/buckets"
    FEATURES_OUT = "/Volumes/profit/feature_store_futures/features" #"/Volumes/profit/feature_store/features"

    FREQ_MIN = 1
    HORIZON_MIN = 60

    # 强烈建议先只跑 1-3 天验证路径 & 输出,不然我怕出问题
    #DATES = ["20251101"]  # 改成 None 会扫全盘，建议你确认后再扫全盘
    DATES = compute_missing_days(BASE_DIR, BUCKETS_OUT, FREQ_MIN)
   # DATES = None
    stage1_build_buckets(BASE_DIR, BUCKETS_OUT, freq_min=FREQ_MIN, dates=DATES)
    
    DATES_STAGE2 = [
    d for d in DATES
    if os.path.exists(os.path.join(BUCKETS_OUT, f"freq={FREQ_MIN}min", f"date={d}", "buckets.parquet"))
    ]
    #stage2_build_features(BUCKETS_OUT, FEATURES_OUT, freq_min=FREQ_MIN, horizon_min=HORIZON_MIN, dates=DATES_STAGE2)
    
    if DATES_STAGE2:
        stage2_build_features(
            BUCKETS_OUT,
            FEATURES_OUT,
            freq_min=FREQ_MIN,
            horizon_min=HORIZON_MIN,
            dates=DATES_STAGE2,
        )
    else:
        print("[Stage2] No newly built bucket files to process.")


[Stage1] SKIP empty day=20251019 (hour_dirs=9 agg_files=0 book_files=0)
[Stage1] SKIP empty day=20251020 (hour_dirs=24 agg_files=0 book_files=0)
[Stage1] SKIP empty day=20251021 (hour_dirs=24 agg_files=0 book_files=0)


FileNotFoundError: No bucket parquet files found for given settings.

In [27]:
BASE_DIR = "/Volumes/profit/bitcoin_ticks/futures/btcusdt"
BUCKETS_OUT = "/Volumes/profit/feature_store_futures/buckets"
FEATURES_OUT = "/Volumes/profit/feature_store_futures/features"

FREQ_MIN = 1
HORIZON_MIN = 60

# 不重跑 Stage1，直接重跑所有已有 buckets 的 Stage2
stage2_build_features(
    BUCKETS_OUT,
    FEATURES_OUT,
    freq_min=FREQ_MIN,
    horizon_min=HORIZON_MIN,
    dates=None,
)

[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251022/features.parquet rows=526
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251023/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251024/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251025/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251026/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251027/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251028/features.parquet rows=1182
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20251029/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/fe

[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260114/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260115/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260116/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260117/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260118/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260119/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260120/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260121/features.parquet rows=1440
[Stage2] wrote /Volumes/profit/f

In [28]:
f = pd.read_parquet("/Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260118/features.parquet")
print("rows:", len(f))
print("unique ts:", f["timestamp"].nunique())
print("dup:", f["timestamp"].duplicated().sum())

rows: 1440
unique ts: 1440
dup: 0


In [ ]:
bpath = "/Volumes/profit/feature_store/buckets/freq=1min/date=20251101/buckets.parquet"
b = pd.read_parquet(bpath)
print("buckets shape:", b.shape)
print("bucket time:", b["timestamp"].min(), b["timestamp"].max())

In [10]:
import io, json, zstandard as zstd

path = "/Volumes/profit/bitcoin_ticks/spot/btcusdt/20260118/00/bookticker/20260118T000037_00001.jsonl.zst"

with open(path, "rb") as f:
    dctx = zstd.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        text = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
        for i, line in enumerate(text):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                print(obj.keys())
                print(obj)
                break
            except Exception:
                continue

dict_keys(['u', 's', 'b', 'B', 'a', 'A', 'market', 'symbol', 'stream', 'local_time_ns', 'best_bid_price', 'best_bid_qty', 'best_ask_price', 'best_ask_qty', 'event_time'])
{'u': 85182974863, 's': 'BTCUSDT', 'b': '95133.81000000', 'B': '4.11562000', 'a': '95133.82000000', 'A': '1.62016000', 'market': 'spot', 'symbol': 'btcusdt', 'stream': 'bookticker', 'local_time_ns': 1768694437490774113, 'best_bid_price': '95133.81000000', 'best_bid_qty': '4.11562000', 'best_ask_price': '95133.82000000', 'best_ask_qty': '1.62016000', 'event_time': 1768694437490}


In [13]:
b = pd.read_parquet("/Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260118/features.parquet")
print(b.shape)
print(b["timestamp"].min(), b["timestamp"].max())
print(b[["mid_last", "spread_mean", "rel_spread_mean", "imbalance_mean"]].isna().mean())
print(b.tail(5))

(1441, 41)
2026-01-18 00:00:00+00:00 2026-01-18 23:59:00+00:00
mid_last           0.001388
spread_mean        0.001388
rel_spread_mean    0.001388
imbalance_mean     0.001388
dtype: float64
          bucket_ms                 timestamp  px_last       px_vwap  vol_qty  \
1436  1768780500000 2026-01-18 23:55:00+00:00  93692.7  93717.004179  255.269   
1437  1768780560000 2026-01-18 23:56:00+00:00  93629.7  93601.902064  716.432   
1438  1768780620000 2026-01-18 23:57:00+00:00  93575.8  93597.656983  445.495   
1439  1768780680000 2026-01-18 23:58:00+00:00  93643.3  93610.031902  266.825   
1440  1768780740000 2026-01-18 23:59:00+00:00  93614.0  93630.612156  147.903   

      vol_notional  n_trades  buy_qty  sell_qty  buy_ratio  ...  spread_60m  \
1436  2.392305e+07    2901.0  178.485    76.784   0.699204  ...         NaN   
1437  6.705940e+07    6504.0  273.082   443.350   0.381169  ...         NaN   
1438  4.169729e+07    2795.0  154.949   290.546   0.347813  ...         NaN   
1439  2

In [15]:
b = pd.read_parquet("/Volumes/profit/feature_store/features/freq=1min/h=60min/date=20260118/features.parquet")
print(b.shape)
print(b["timestamp"].min(), b["timestamp"].max())
print(b[["mid_last", "spread_mean", "rel_spread_mean", "imbalance_mean"]].isna().mean())
print(b.tail(5))

print("n_rows:", len(b))
print("n_unique_ts:", b["timestamp"].nunique())
print("dup_count:", b["timestamp"].duplicated().sum())

dups = b[b["timestamp"].duplicated(keep=False)].sort_values("timestamp")
print(dups[["timestamp", "bucket_ms"]].head(20))

(1441, 41)
2026-01-18 00:00:00+00:00 2026-01-18 23:59:00+00:00
mid_last           0.0
spread_mean        0.0
rel_spread_mean    0.0
imbalance_mean     0.0
dtype: float64
          bucket_ms                 timestamp   px_last       px_vwap  \
1436  1768780500000 2026-01-18 23:55:00+00:00  93740.04  93760.009156   
1437  1768780560000 2026-01-18 23:56:00+00:00  93690.32  93677.952564   
1438  1768780620000 2026-01-18 23:57:00+00:00  93629.31  93651.584755   
1439  1768780680000 2026-01-18 23:58:00+00:00  93696.33  93674.077191   
1440  1768780740000 2026-01-18 23:59:00+00:00  93673.14  93686.391839   

       vol_qty  vol_notional  n_trades   buy_qty  sell_qty  buy_ratio  ...  \
1436  10.44110  9.789576e+05    1392.0   7.07006   3.37104   0.677137  ...   
1437  66.78626  6.256400e+06    3463.0  16.28274  50.50352   0.243804  ...   
1438  15.25597  1.428746e+06    1369.0   8.42286   6.83311   0.552103  ...   
1439  19.18029  1.796696e+06    1175.0  15.57076   3.60953   0.811810  ...   
1

In [16]:
b_buckets = pd.read_parquet("/Volumes/profit/feature_store_futures/buckets/freq=1min/date=20260118/buckets.parquet")
print("BUCKETS")
print("rows:", len(b_buckets))
print("unique ts:", b_buckets["timestamp"].nunique())
print("dup:", b_buckets["timestamp"].duplicated().sum())
print(b_buckets[b_buckets["timestamp"].duplicated(keep=False)][["timestamp", "bucket_ms"]])

BUCKETS
rows: 1441
unique ts: 1441
dup: 0
Empty DataFrame
Columns: [timestamp, bucket_ms]
Index: []


In [17]:
b_feat = pd.read_parquet("/Volumes/profit/feature_store_futures/features/freq=1min/h=60min/date=20260118/features.parquet")
print("FEATURES")
print("rows:", len(b_feat))
print("unique ts:", b_feat["timestamp"].nunique())
print("dup:", b_feat["timestamp"].duplicated().sum())
print(b_feat[b_feat["timestamp"].duplicated(keep=False)][["timestamp", "bucket_ms"]])

FEATURES
rows: 1441
unique ts: 1440
dup: 1
                  timestamp      bucket_ms
0 2026-01-18 00:00:00+00:00  1768694400000
1 2026-01-18 00:00:00+00:00  1768694400000
